# Silver: Representantes

**Objetivo:** transformar a tabela Bronze de Representantes em uma tabela Silver confiável e versionada, aplicando:
1. Padronização do nome: remoção da sujeira proposital do simulador (maiúsculas aleatórias, espaços extras nas bordas) e remoção de acentos
2. Controle SCD Tipo 2 real, via MERGE

**Fonte:** tabela Delta `bronze.representantes`.

**Destino:** tabela Delta `silver.representantes`.

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, upper, initcap, trim, translate, regexp_replace , current_date, lit
from delta.tables import DeltaTable

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.silver")

print("Schema 'silver' verificado/criado com sucesso.")

In [0]:
# Padronização agressiva - remoção de títulos, acentos, maiúsculas em tudo

import re

caracteres_com_acento = "áàãâäéèêëíìîïóòõôöúùûüçñÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
caracteres_sem_acento = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

df_bronze_representantes = spark.table("poc_latam_food.bronze.representantes")

df_representantes_padronizado = (
    df_bronze_representantes
    # Remove títulos comuns (Sr., Sra., Srta., Dr., Dra., Lic., Ing.) antes de qualquer outra limpeza
    .withColumn("nome", regexp_replace(col("nome"), r"(?i)\b(sr|sra|srta|dr|dra|lic|ing)\.?\s+", ""))
    .withColumn("nome", trim(col("nome")))
    .withColumn("nome", upper(translate(col("nome"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("cargo", upper(translate(col("cargo"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn("pais", upper(translate(col("pais"), caracteres_com_acento, caracteres_sem_acento)))
    .withColumn(
        "centro_vinculado",
        upper(translate(col("centro_vinculado"), caracteres_com_acento, caracteres_sem_acento))
    )
    .select("representante_id", "nome", "cargo", "pais", "centro_vinculado")
)

df_representantes_padronizado.display()

In [0]:
# SCD2 real via MERGE

tabela_existe = spark.catalog.tableExists("poc_latam_food.silver.representantes")

if not tabela_existe:
    df_primeira_carga = (
        df_representantes_padronizado
        .withColumn("data_inicio", current_date())
        .withColumn("data_fim", lit(None).cast("date"))
        .withColumn("flag_ativo", lit(True))
    )
    df_primeira_carga.write.format("delta").saveAsTable("poc_latam_food.silver.representantes")
    print("Carga inicial da Silver de Representantes concluída.")

else:
    tabela_silver = DeltaTable.forName(spark, "poc_latam_food.silver.representantes")

    colunas_comparadas = ["nome", "cargo", "pais", "centro_vinculado"]
    condicao_mudanca = " OR ".join([
        f"target.{c} <> source.{c}" for c in colunas_comparadas
    ])

    (
        tabela_silver.alias("target")
        .merge(
            df_representantes_padronizado.alias("source"),
            "target.representante_id = source.representante_id AND target.flag_ativo = true"
        )
        .whenMatchedUpdate(
            condition=condicao_mudanca,
            set={"flag_ativo": lit(False), "data_fim": current_date()}
        )
        .execute()
    )

    df_vigentes = spark.table("poc_latam_food.silver.representantes").filter("flag_ativo = true")

    df_para_inserir = (
        df_representantes_padronizado.alias("source")
        .join(df_vigentes.alias("target"), on="representante_id", how="left_anti")
        .withColumn("data_inicio", current_date())
        .withColumn("data_fim", lit(None).cast("date"))
        .withColumn("flag_ativo", lit(True))
    )

    df_para_inserir.write.format("delta").mode("append").saveAsTable("poc_latam_food.silver.representantes")

    print(f"MERGE concluído: {df_para_inserir.count()} nova(s) versão(ões) inserida(s).")

In [0]:
# Validação visual - silver.representantes

spark.table("poc_latam_food.silver.representantes").display()

print(f"Total de representantes: {spark.table('poc_latam_food.silver.representantes').count()}")